In [ ]:
import os
import json
import time
import re
import requests
import pandas as pd
import numpy as np
import glob
import asyncio
import random
from pathlib import Path
from typing import Dict, List, Optional, Union, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, HTML
from dotenv import load_dotenv
import sys

# Добавляем путь к модулям проекта
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
ml_path = project_root / "ml"

from ml.models.baseline import HRBaseline, langfuse

# Загружаем ключи
load_dotenv()
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
MISTRAL_URL = "https://api.mistral.ai/v1/chat/completions"
MISTRAL_MODEL = "mistral-large-latest"

# Инициализируем модели
hr_system = HRBaseline()

def clean_mistral_output(output: str) -> str:
    if not isinstance(output, str):
        return ""
    clean = re.sub(r"^```(?:json)?", "", output.strip(), flags=re.IGNORECASE | re.MULTILINE)
    clean = re.sub(r"```$", "", clean.strip(), flags=re.MULTILINE)
    clean = re.sub(r"\n\s*-\s*\n", "\n", clean)
    clean = re.sub(r"^\s*-\s*{", "{", clean, flags=re.MULTILINE)
    clean = re.sub(r'[\x00-\x1f\x7f]', ' ', clean)
    end = clean.rfind("}")
    if end != -1:
        clean = clean[:end+1]
    return clean.strip()

def post_with_retries(url, headers, payload, timeout=60, max_retries=5, base_delay=0.8):
    """Ретраи с экспоненциальной задержкой и джиттером для 429/5xx ошибок."""
    last_resp = None
    for attempt in range(1, max_retries + 1):
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        last_resp = resp

        if resp.status_code == 200:
            return resp

        retry_after = resp.headers.get("Retry-After")
        if resp.status_code in (429, 500, 502, 503, 504):
            if attempt == max_retries:
                break
            if retry_after:
                try:
                    delay = float(retry_after)
                except Exception:
                    delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            else:
                delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            time.sleep(delay)
            continue
        break
    return last_resp
    
def _find_test_files(test_folders):

    """Ищет тестовые файлы в различных папках"""
    test_files = []
    for folder in test_folders:
        if os.path.exists(folder):
            # Ищем все поддерживаемые форматы
            patterns = [f"{folder}/*.pdf", f"{folder}/*.docx", f"{folder}/*.doc", f"{folder}/*.txt"]
            for pattern in patterns:
                found_files = glob.glob(pattern)
                test_files.extend(found_files)
    
    return test_files

def check_extraction_with_mistral(
   resume_files,  #Исходные файлы с резюме
   extracted_data #Извлеченные из резюме данные
):
    """
    Проверяет:
    
    """
    headers = {
            "Authorization": f"Bearer {MISTRAL_API_KEY}",
            "Content-Type": "application/json"
        }
    
    prompt = (
        "Ты проверяющий ассистент. Верни строго валидный JSON без Markdown.\n\n"
        "Задача: проверить извлеченные из резюме данные на соответсвие исходным данным и на присутствие необходимых полей.\n"
        "Необходимые поля:\n"
        " - contacts\n"
        " - skills\n"
        " - experience\n"
        " - education\n\n"
        f"Исходный файл резюме:\n{resume_files}\n\n"
        f"Извлеченные данные из резюме: {json.dumps(extracted_data, ensure_ascii=False, indent=2)}\n"
        "Проверь:\n"
        "1) Присутствуют ли все необходивые поля.\n"
        "2) Все ли навыки были извлечены.\n"
        "3) Все ли места работы извлеклись\n"
        "4) Общую оценку извлечения данных. На сколько извлеченные данные соответствуют оригинальному резюме (из 100).\n\n"
        "Ответь ТОЛЬКО JSON следующей формы:\n"
        "{\n"
        '  "all_fields_present": true,\n'
        '  "all_skills_present": true,\n'
        '  "all_experience_companies": true,\n'
        '  "match_analysis": 100\n'
        "}\n"
    )

    payload = {
            "model": MISTRAL_MODEL,
            "messages": [
            {"role": "system", "content": "Ты проверяющий ассистент. Возвращай только JSON."},
            {"role": "user", "content": prompt}
        ],
            "temperature": 0.1,
            "max_tokens": 1024,
        }
    
    try:
        resp = requests.post(MISTRAL_URL, headers=headers, json=payload, timeout=60)
        if resp.status_code != 200:
            return {"error": f"Mistral API error: {resp.status_code}"}
        
        data = resp.json()
        content = (
            data.get("choices", [{}])[0]
            .get("message", {})
            .get("content", "")
            .strip()
        )
        
        if not content:
            return {"error": "Empty response"}
        
        cleaned = clean_mistral_output(content)
        try:
            parsed = json.loads(cleaned)
        except Exception as e:
            return {"error": f"Invalid JSON: {e}", "raw_output": content}
            
    except Exception as e:
        return {"error": f"API call failed: {e}"}

    usage = data.get("usage", {}) or {}
    return {
        "all_fields_present": bool(parsed.get("all_fields_present", False)),
        "all_skills_present": bool(parsed.get("all_skills_present", False)),
        "all_experience_companies": bool(parsed.get("all_experience_companies", False)),
        "match_analysis": int(parsed.get("match_analysis", 0)),
        "usage": usage
    }

def check_evaluation_with_mistral(
   resume_data,               #Файлы с данными из резюме
   vacancy,                   #Вакансия
   resume__with_evaluation,   #Файлы с оценками резюме
):
    """
    Проверяет:
    
    """
    headers = {
            "Authorization": f"Bearer {MISTRAL_API_KEY}",
            "Content-Type": "application/json"
        }
    
    prompt = (
        "Ты проверяющий ассистент. Верни строго валидный JSON без Markdown.\n\n"
        "Задача: проверить оценки резюме по соответствию вакансии и веса критериев оценки на адекватность и объективность. \n"
        "В полях будут оценки того, на сколько по этому критерию резюме кандидата соответствует вакансии.\n"
        "Веса критериев в сумме должны давать 1.\n"
        f"Файл с данными из резюме:\n{json.dumps(resume_data, ensure_ascii=False, indent=2)}\n\n"
        f"Файл с вакансией: {json.dumps(vacancy, ensure_ascii=False, indent=2)}\n"
        f"Файл с оценками резюме:\n{json.dumps(resume__with_evaluation, ensure_ascii=False, indent=2)}\n\n"
        "Проверь:\n"
        "1) На сколько оценка в поле job_title_match точна (от 0 до 100).\n"
        "2) Какой вес в поле job_title_match лучше всего подходит для данного критерия (от 0 до 1).\n"
        "3) На сколько оценка в поле education_match точна (от 0 до 100).\n"
        "4) Какой вес в поле education_match лучше всего подходит для данного критерия (от 0 до 1).\n"
        "5) На сколько оценка в поле experience_match точна (от 0 до 100).\n"
        "6) Какой вес в поле experience_match лучше всего подходит для данного критерия (от 0 до 1).\n"
        "7) На сколько оценка в поле schedule_match точна (от 0 до 100).\n"
        "8) Какой вес в поле schedule_match лучше всего подходит для данного критерия (от 0 до 1).\n"
        "9) На сколько оценка в поле format_match точна (от 0 до 100).\n"
        "10) Какой вес в поле format_match лучше всего подходит для данного критерия (от 0 до 1).\n"
        "11) На сколько оценка в поле additional_match точна (от 0 до 100).\n"
        "12) Какой вес в поле additional_match лучше всего подходит для данного критерия (от 0 до 1).\n"
        "13) На сколько общая оценка соответствия резюме вакансии (overall_score) соответствует действительности (от 0 до 100).\n"
        "14) На сколько поле recommendation соответствует резюме относительно вакансии (от 0 до 100).\n"
        "Ответь ТОЛЬКО JSON следующей формы:\n"
        "{\n"
        '  "job_title_match_eval": 100,\n'
        '  "job_title_match_weight": 0.5,\n'
        '  "education_match_eval": 100,\n'
        '  "education_match_weight": 0.5,\n'
        '  "experience_match_eval": 100,\n'
        '  "experience_match_weight": 0.5,\n'
        '  "schedule_match_eval": 100,\n'
        '  "schedule_match_weight": 0.5,\n'
        '  "format_match_eval": 100,\n'
        '  "format_match_weight": 0.5,\n'
        '  "additional_match_eval": 100,\n'
        '  "additional_match_weight": 0.5,\n'
        '  "overall_score_estimation": 100,\n'
        '  "recomendation_estimation": 100\n'
        "}\n"
    )

    payload = {
            "model": MISTRAL_MODEL,
            "messages": [
            {"role": "system", "content": "Ты проверяющий ассистент. Возвращай только JSON."},
            {"role": "user", "content": prompt}
        ],
            "temperature": 0.1,
            "max_tokens": 1024,
        }
    
    try:
        resp = requests.post(MISTRAL_URL, headers=headers, json=payload, timeout=60)
        if resp.status_code != 200:
            return {"error": f"Mistral API error: {resp.status_code}"}
        
        data = resp.json()
        content = (
            data.get("choices", [{}])[0]
            .get("message", {})
            .get("content", "")
            .strip()
        )
        
        if not content:
            return {"error": "Empty response"}
        
        cleaned = clean_mistral_output(content)
        try:
            parsed = json.loads(cleaned)
        except Exception as e:
            return {"error": f"Invalid JSON: {e}", "raw_output": content}
            
    except Exception as e:
        return {"error": f"API call failed: {e}"}

    usage = data.get("usage", {}) or {}
    return {
        "job_title_match_eval": int(parsed.get("job_title_match_eval", 0)),
        "job_title_match_weight": float(parsed.get("job_title_match_weight", 0)),
        "education_match_eval": int(parsed.get("education_match_eval", 0)),
        "education_match_weight": float(parsed.get("education_match_weight", 0)),
        "experience_match_eval": int(parsed.get("experience_match_eval", 0)),
        "experience_match_weight": float(parsed.get("experience_match_weight", 0)),
        "schedule_match_eval": int(parsed.get("schedule_match_eval", 0)),
        "schedule_match_weight": float(parsed.get("schedule_match_weight", 0)),
        "format_match_eval": int(parsed.get("format_match_eval", 0)),
        "format_match_weight": float(parsed.get("format_match_weight", 0)),
        "additional_match_eval": int(parsed.get("additional_match_eval", 0)),
        "additional_match_weight": float(parsed.get("additional_match_weight", 0)),
        "overall_score_estimation": int(parsed.get("overall_score_estimation", 0)),
        "recomendation_estimation": int(parsed.get("recomendation_estimation", 0)),
        "usage": usage
    }

def check_questions_with_mistral(
    extracted_data,               # Файлы с данными из резюме
    vacancy,                      # Вакансия
    questions_for_resume,         # Файлы с вопросами
):
    """
    Проверяет сгенерированные вопросы на:
    1. Релевантность данным резюме
    2. Релевантность требованиям вакансии
    3. Качество вопросов (конкретность, полезность для оценки)
    4. Разнообразие вопросов (не повторяются)
    """
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    
    # Проверяем структуру вопросов
    if isinstance(questions_for_resume, list):
        questions_list = questions_for_resume
    elif isinstance(questions_for_resume, dict):
        if "questions" in questions_for_resume:
            questions_list = questions_for_resume["questions"]
        elif "generated_questions" in questions_for_resume:
            questions_list = questions_for_resume["generated_questions"]
        else:
            questions_list = list(questions_for_resume.values())
    else:
        return {"error": "Неверный формат вопросов"}
    
    prompt = (
        "Ты проверяющий ассистент для HR-системы. Верни строго валидный JSON без Markdown.\n\n"
        "Задача: проверить качество сгенерированных вопросов для кандидата на основе его резюме и вакансии.\n\n"
        "КРИТЕРИИ ПРОВЕРКИ:\n"
        "1. Релевантность резюме: Вопросы должны быть основаны на данных из резюме\n"
        "2. Релевантность вакансии: Вопросы должны помогать оценить соответствие требованиям вакансии\n"
        "3. Конкретность: Вопросы должны быть конкретными, а не общими\n"
        "4. Разнообразие: Вопросы не должны повторяться и должны охватывать разные аспекты\n"
        "5. Практическая полезность: Вопросы должны помогать принять решение о найме\n\n"
        "ДАННЫЕ РЕЗЮМЕ:\n"
        f"{extracted_data}\n\n"
        "ДАННЫЕ ВАКАНСИИ:\n"
        f"{json.dumps(vacancy, ensure_ascii=False, indent=2)}\n\n"
        "СГЕНЕРИРОВАННЫЕ ВОПРОСЫ:\n"
        f"{questions_list}\n\n"
        "ПРОВЕРЬ:\n"
        "1) Релевантность_резюме: Насколько вопросы соответствуют данным из резюме (от 0 до 100)\n"
        "2) Релевантность_вакансии: Насколько вопросы помогают оценить соответствие требованиям вакансии (от 0 до 100)\n"
        "3) Конкретность_вопросов: Насколько вопросы конкретны и сфокусированы (от 0 до 100)\n"
        "4) Разнообразие_вопросов: Насколько вопросы разнообразны и покрывают разные темы (от 0 до 100)\n"
        "5) Практическая_полезность: Насколько вопросы полезны для принятия решения о найме (от 0 до 100)\n"
        "6) Количество_повторений: Сколько раз повторяются похожие вопросы (число)\n"
        "7) Рекомендации: Список рекомендаций по улучшению вопросов\n"
        "8) Лучшие_вопросы: Список 3-х лучших вопросов из предоставленных\n"
        "9) Вопросы_на_улучшение: Список 3-х вопросов, которые нужно улучшить или заменить\n\n"
        "Ответь ТОЛЬКО JSON следующей формы:\n"
        "{\n"
        '  "relevance_to_resume": 85,\n'
        '  "relevance_to_vacancy": 90,\n'
        '  "question_specificity": 75,\n'
        '  "question_diversity": 80,\n'
        '  "practical_usefulness": 88,\n'
        '  "repetition_count": 2,\n'
        '  "recommendations": ["рекомендация 1", "рекомендация 2"],\n'
        '  "best_questions": ["лучший вопрос 1", "лучший вопрос 2", "лучший вопрос 3"],\n'
        '  "questions_to_improve": ["вопрос на улучшение 1", "вопрос на улучшение 2", "вопрос на улучшение 3"]\n'
        "}\n"
    )

    payload = {
        "model": MISTRAL_MODEL,
        "messages": [
            {"role": "system", "content": "Ты опытный HR-эксперт, который оценивает качество собеседования. Возвращай только JSON."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.1,
        "max_tokens": 2048,
    }
    
    try:
        resp = requests.post(MISTRAL_URL, headers=headers, json=payload, timeout=90)
        if resp.status_code != 200:
            return {"error": f"Mistral API error: {resp.status_code}"}
        
        data = resp.json()
        content = (
            data.get("choices", [{}])[0]
            .get("message", {})
            .get("content", "")
            .strip()
        )
        
        if not content:
            return {"error": "Empty response"}
        
        cleaned = clean_mistral_output(content)
        try:
            parsed = json.loads(cleaned)
        except Exception as e:
            return {"error": f"Invalid JSON: {e}", "raw_output": content}
            
    except Exception as e:
        return {"error": f"API call failed: {e}"}

    usage = data.get("usage", {}) or {}
    
    return {
        "relevance_to_resume": int(parsed.get("relevance_to_resume", 0)),
        "relevance_to_vacancy": int(parsed.get("relevance_to_vacancy", 0)),
        "question_specificity": int(parsed.get("question_specificity", 0)),
        "question_diversity": int(parsed.get("question_diversity", 0)),
        "practical_usefulness": int(parsed.get("practical_usefulness", 0)),
        "repetition_count": int(parsed.get("repetition_count", 0)),
        "recommendations": parsed.get("recommendations", []),
        "best_questions": parsed.get("best_questions", []),
        "questions_to_improve": parsed.get("questions_to_improve", []),
        "usage": usage,
        "total_questions_analyzed": len(questions_list)
    }

async def run_test_extraction():

    passed_all = passed_fields = passed_skills = passed_experience = passed_experience_name = passed_experience_description = passed_match_analysis = 0
    all_results = []

    test_files = _find_test_files([
        "data/test_resumes"
    ])
    
    total = len(test_files)
    total_match = 0
    
    for i, file_path in enumerate(test_files, 1):
        print(f"\n--- Файл {i}/{len(test_files)}: {os.path.basename(file_path)} ---")
        
        start = time.time()
        # Извелкаем данные из резюме
        extracted_data = await hr_system.extract_data_from_resume(file_path)

        gen_latency = round(time.time() - start, 3)

        check_extraction = check_extraction_with_mistral(file_path, extracted_data)

        all_fields_present = "OK" if check_extraction.get("all_fields_present", False) else "FAIL"
        all_skills_present = "OK" if check_extraction.get("all_skills_present", False) else "FAIL"
        all_experience_companies = "OK" if check_extraction.get("all_experience_companies", False) else "FAIL"
        match_analysis = check_extraction.get("match_analysis")
        usage = check_extraction.get("usage", {}) or {}

        if all_fields_present == "OK": passed_fields += 1
        if all_skills_present == "OK": passed_skills += 1
        if all_experience_companies == "OK": passed_experience += 1
        if match_analysis >= 70 : 
            passed_match_analysis += 1
            total_match += match_analysis
        if all(v == "OK" for v in [all_fields_present, all_skills_present, all_experience_companies]) and match_analysis >= 70:
            passed_all += 1

        print(f"  Все поля: {all_fields_present}, Все навыки: {all_skills_present}, Места работы: {all_experience_companies}, Обшая оценка извлечения: {match_analysis}")
 
        all_results.append({
                "id": i,
                "all_fields_present": all_fields_present,
                "all_skills_present": all_skills_present,
                "all_experience_companies": all_experience_companies,
                "match_analysis": match_analysis,
                "gen_latency": gen_latency,
                "total_tokens": usage.get("total_tokens"),
                "error": None
            })

    # Закрываем клиент
    await hr_system.close_client()

    print(f"\n📊 Итог по {total} тестам:")
    print(f"✅ Все ли поля на месте: {passed_fields}/{total}")
    print(f"✅ Все ли навыки извлечены: {passed_skills}/{total}")
    print(f"✅ Весь ли опыт извлечен: {all_experience_companies}/{total}")
    print(f"✅ Оценки извлечения: {passed_match_analysis}/{total}")
    print(f"✅ Средняя оценка извлечения: {total_match / total}")
    print(f"✅ Все условия: {passed_all}/{total}")

async def run_test_evaluation():

    all_results = []

    test_files = _find_test_files([
        "data/test_resumes"
    ])

    vacancy = "data/vacancy.json"

    # Загружаем данные вакансии (один раз)
    try:
        with open(vacancy, 'r', encoding='utf-8') as f:
            vacancy = json.load(f)
        print(f"✅ Загружены данные вакансии: {vacancy.get('position', 'Не указана')}")
    except Exception as e:
        print(f"❌ Ошибка загрузки данных вакансии: {e}")
        return
    
    for i, file_path in enumerate(test_files, 1):
        print(f"\n--- Файл {i}/{len(test_files)}: {os.path.basename(file_path)} ---")
    
        start = time.time()

        # Извелкаем данные из резюме
        extracted_data = await hr_system.extract_data_from_resume(file_path)
        
        #Оцениваем резюме
        eval_resumes = await hr_system.evaluate_candidate_match(extracted_data, vacancy)

        gen_latency = round(time.time() - start, 3)

        #Оцениваем оценки резюме
        check_evaluation = check_evaluation_with_mistral(file_path, vacancy, eval_resumes)

        job_title_match_eval = check_evaluation.get("job_title_match_eval") 
        job_title_match_weight = check_evaluation.get("job_title_match_weight")
        education_match_eval = check_evaluation.get("education_match_eval") 
        education_match_weight = check_evaluation.get("education_match_weight")
        experience_match_eval = check_evaluation.get("experience_match_eval") 
        experience_match_weight = check_evaluation.get("experience_match_weight")
        schedule_match_eval = check_evaluation.get("schedule_match_eval") 
        schedule_match_weight = check_evaluation.get("schedule_match_weight")
        format_match_eval = check_evaluation.get("format_match_eval") 
        format_match_weight = check_evaluation.get("format_match_weight")
        additional_match_eval = check_evaluation.get("additional_match_eval") 
        additional_match_weight = check_evaluation.get("additional_match_weight")
        overall_score_estimation = check_evaluation.get("overall_score_estimation")
        recomendation_estimation = check_evaluation.get("recomendation_estimation")
        usage = check_evaluation.get("usage", {}) or {}

        print(f"Соответствие должности: {job_title_match_eval}\n"
                f"Вес криетрия должности:{job_title_match_weight}\n"
                f"Соответствие уровня образования: {education_match_eval}\n"
                f"Вес криетрия уровня образования: {education_match_weight}\n"
                f"Соответствие опыту: {experience_match_eval}\n"
                f"Вес криетрия опыта: {experience_match_weight}\n"
                f"Соответствие графику работы: {schedule_match_eval}\n"
                f"Вес криетрия графика работы: {schedule_match_weight}\n"
                f"Соответствие формату работы: {format_match_eval}\n"
                f"Вес криетрия формата работы: {format_match_weight}\n"
                f"Соответствие дополнительным критериям: {additional_match_eval}\n"
                f"Вес криетрия дополнительных критерией: {additional_match_weight}\n"
                f"Общая оценка: {overall_score_estimation}\n"
                f"Соотвествие рекомедаций: {recomendation_estimation}\n"
               )
 
        all_results.append({
                "id": i,
                "job_title_match_eval": job_title_match_eval,
                "job_title_match_weight": job_title_match_weight,
                "education_match_eval": education_match_eval,
                "education_match_weight": education_match_weight,
                "experience_match_eval": experience_match_eval,
                "experience_match_weight": experience_match_weight,
                "schedule_match_eval": schedule_match_eval,
                "schedule_match_weight": schedule_match_weight,
                "format_match_eval": format_match_eval,
                "format_match_weight": format_match_weight,
                "additional_match_eval": additional_match_eval,
                "additional_match_weight": additional_match_weight,
                "overall_score_estimation": overall_score_estimation,
                "recomendation_estimation": recomendation_estimation,
                "gen_latency": gen_latency,
                "total_tokens": usage.get("total_tokens"),
                "error": None
            })
        
    # Закрываем клиент
    await hr_system.close_client()

async def run_test_question():
    """Тестирование генерации вопросов к резюме"""
    
    # Инициализация системы
    hr_system = HRBaseline()
    
    test_files = _find_test_files([
        "data/test_resumes"
    ])

    vacancy = "data/vacancy.json"
    
    # Загружаем данные вакансии (один раз)
    try:
        with open(vacancy, 'r', encoding='utf-8') as f:
            vacancy = json.load(f)
        print(f"✅ Загружены данные вакансии: {vacancy.get('position', 'Не указана')}")
    except Exception as e:
        print(f"❌ Ошибка загрузки данных вакансии: {e}")
        return
    
    passed_all = passed_relevance_resume = passed_relevance_vacancy = passed_specificity = passed_diversity = passed_usefulness = 0
    all_results = []
    
    total = len(test_files)
    total_relevance_resume = 0
    total_relevance_vacancy = 0
    total_specificity = 0
    total_diversity = 0
    total_usefulness = 0
    
    for i, file_path in enumerate(test_files, 1):
            
        print(f"\n--- Файл {i}/{len(test_files)}: {os.path.basename(file_path)} ---")
        
        start = time.time()

        # Загружаем данные
        try:
           # Извелкаем данные из резюме
            extracted_data = await hr_system.extract_data_from_resume(file_path)
            
            #Оцениваем резюме
            question_resumes = await hr_system.generate_interview_questions(extracted_data, vacancy)

            gen_latency = round(time.time() - start, 3)

        except Exception as e:
            print(f"❌ Ошибка загрузки данных: {e}")
            continue
        
        # Проверяем качество вопросов
        check_result = check_questions_with_mistral(
            extracted_data=extracted_data,
            vacancy=vacancy,
            questions_for_resume=question_resumes
        )
        
        if "error" in check_result:
            print(f"❌ Ошибка проверки: {check_result['error']}")
            all_results.append({
                "id": i,
                "resume_file": os.path.basename(file_path),
                "relevance_to_resume": "FAIL",
                "relevance_to_vacancy": "FAIL",
                "question_specificity": "FAIL",
                "question_diversity": "FAIL",
                "practical_usefulness": "FAIL",
                "repetition_count": 0,
                "gen_latency": gen_latency,
                "total_tokens": 0,
                "error": check_result.get("error"),
                "total_questions": 0
            })
            continue
        
        # Извлекаем метрики
        relevance_resume_score = check_result.get("relevance_to_resume", 0)
        relevance_vacancy_score = check_result.get("relevance_to_vacancy", 0)
        specificity_score = check_result.get("question_specificity", 0)
        diversity_score = check_result.get("question_diversity", 0)
        usefulness_score = check_result.get("practical_usefulness", 0)
        repetition_count = check_result.get("repetition_count", 0)
        usage = check_result.get("usage", {}) or {}
        
        # Определяем статусы OK/FAIL (порог 70%)
        relevance_resume_ok = "OK" if relevance_resume_score >= 70 else "FAIL"
        relevance_vacancy_ok = "OK" if relevance_vacancy_score >= 70 else "FAIL"
        specificity_ok = "OK" if specificity_score >= 70 else "FAIL"
        diversity_ok = "OK" if diversity_score >= 70 else "FAIL"
        usefulness_ok = "OK" if usefulness_score >= 70 else "FAIL"
        
        # Считаем статистику
        if relevance_resume_ok == "OK": 
            passed_relevance_resume += 1
            total_relevance_resume += relevance_resume_score
        
        if relevance_vacancy_ok == "OK": 
            passed_relevance_vacancy += 1
            total_relevance_vacancy += relevance_vacancy_score
        
        if specificity_ok == "OK": 
            passed_specificity += 1
            total_specificity += specificity_score
        
        if diversity_ok == "OK": 
            passed_diversity += 1
            total_diversity += diversity_score
        
        if usefulness_ok == "OK": 
            passed_usefulness += 1
            total_usefulness += usefulness_score
        
        # Проверяем все условия
        all_conditions_ok = all(v == "OK" for v in [
            relevance_resume_ok, relevance_vacancy_ok, 
            specificity_ok, diversity_ok, usefulness_ok
        ])
        
        if all_conditions_ok:
            passed_all += 1
        
        print(f"📊 Результаты проверки:")
        print(f"  • Релевантность резюме: {relevance_resume_score}/100 ({relevance_resume_ok})")
        print(f"  • Релевантность вакансии: {relevance_vacancy_score}/100 ({relevance_vacancy_ok})")
        print(f"  • Конкретность вопросов: {specificity_score}/100 ({specificity_ok})")
        print(f"  • Разнообразие вопросов: {diversity_score}/100 ({diversity_ok})")
        print(f"  • Практическая полезность: {usefulness_score}/100 ({usefulness_ok})")
        print(f"  • Повторений: {repetition_count}")
        print(f"  • Время проверки: {gen_latency} сек")
        
        # Показываем лучшие вопросы
        best_questions = check_result.get("best_questions", [])
        if best_questions:
            print(f"\n🏆 Лучшие вопросы:")
            for j, question in enumerate(best_questions[:3], 1):
                print(f"  {j}. {question}")
        
        # Показываем вопросы для улучшения
        questions_to_improve = check_result.get("questions_to_improve", [])
        if questions_to_improve:
            print(f"\n⚠️ Вопросы для улучшения:")
            for j, question in enumerate(questions_to_improve[:3], 1):
                print(f"  {j}. {question}")
        
        # Сохраняем результат
        all_results.append({
            "id": i,
            "resume_file": os.path.basename(file_path),
            "relevance_to_resume": relevance_resume_ok,
            "relevance_to_resume_score": relevance_resume_score,
            "relevance_to_vacancy": relevance_vacancy_ok,
            "relevance_to_vacancy_score": relevance_vacancy_score,
            "question_specificity": specificity_ok,
            "question_specificity_score": specificity_score,
            "question_diversity": diversity_ok,
            "question_diversity_score": diversity_score,
            "practical_usefulness": usefulness_ok,
            "practical_usefulness_score": usefulness_score,
            "repetition_count": repetition_count,
            "gen_latency": gen_latency,
            "total_tokens": usage.get("total_tokens"),
            "error": None,
            "total_questions": check_result.get("total_questions_analyzed", 0),
            "best_questions": best_questions[:3],
            "questions_to_improve": questions_to_improve[:3]
        })
    
    # Закрываем клиент
    await hr_system.close_client()
    
    # Выводим итоговую статистику
    print(f"\n{'='*60}")
    print("📊 ИТОГОВАЯ СТАТИСТИКА ПО ГЕНЕРАЦИИ ВОПРОСОВ")
    print(f"{'='*60}")
    print(f"Всего протестировано: {total} резюме")
    print(f"\n✅ Релевантность резюме: {passed_relevance_resume}/{total}")
    if passed_relevance_resume > 0:
        print(f"   Средняя оценка: {total_relevance_resume/passed_relevance_resume:.1f}/100")
    
    print(f"\n✅ Релевантность вакансии: {passed_relevance_vacancy}/{total}")
    if passed_relevance_vacancy > 0:
        print(f"   Средняя оценка: {total_relevance_vacancy/passed_relevance_vacancy:.1f}/100")
    
    print(f"\n✅ Конкретность вопросов: {passed_specificity}/{total}")
    if passed_specificity > 0:
        print(f"   Средняя оценка: {total_specificity/passed_specificity:.1f}/100")
    
    print(f"\n✅ Разнообразие вопросов: {passed_diversity}/{total}")
    if passed_diversity > 0:
        print(f"   Средняя оценка: {total_diversity/passed_diversity:.1f}/100")
    
    print(f"\n✅ Практическая полезность: {passed_usefulness}/{total}")
    if passed_usefulness > 0:
        print(f"   Средняя оценка: {total_usefulness/passed_usefulness:.1f}/100")
    
    print(f"\n🎯 Все критерии выполнены: {passed_all}/{total}")
    
    # Общая средняя оценка по всем успешным тестам
    successful_tests = [r for r in all_results if r.get("error") is None]
    if successful_tests:
        avg_scores = {
            "relevance_resume": sum(r.get("relevance_to_resume_score", 0) for r in successful_tests) / len(successful_tests),
            "relevance_vacancy": sum(r.get("relevance_to_vacancy_score", 0) for r in successful_tests) / len(successful_tests),
            "specificity": sum(r.get("question_specificity_score", 0) for r in successful_tests) / len(successful_tests),
            "diversity": sum(r.get("question_diversity_score", 0) for r in successful_tests) / len(successful_tests),
            "usefulness": sum(r.get("practical_usefulness_score", 0) for r in successful_tests) / len(successful_tests)
        }
        
        print(f"\n📈 Общие средние оценки:")
        print(f"  • Релевантность резюме: {avg_scores['relevance_resume']:.1f}/100")
        print(f"  • Релевантность вакансии: {avg_scores['relevance_vacancy']:.1f}/100")
        print(f"  • Конкретность: {avg_scores['specificity']:.1f}/100")
        print(f"  • Разнообразие: {avg_scores['diversity']:.1f}/100")
        print(f"  • Полезность: {avg_scores['usefulness']:.1f}/100")
    
    return all_results



#asyncio.run(run_test_extraction())
#asyncio.run(run_test_evaluation())
asyncio.run(run_test_question())

<coroutine object run_tests_extrcutcion at 0x00000246A1731480>

In [ ]:
PS C:\Users\67181\OneDrive\Dokumenty\bachelor-2025-team-team> & C:/Users/67181/AppData/Local/Programs/Python/Python312/python.exe c:/Users/67181/OneDrive/Dokumenty/bachelor-2025-team-team/ml/evaluation/test_with_llm.py

--- Файл 1/10: 1.docx ---
📄 Парсим файл...
✅ Извлечено 3702 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: FAIL, Места работы: FAIL, Обшая оценка извлечения: 85

--- Файл 2/10: 10.docx ---
📄 Парсим файл...
✅ Извлечено 3740 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: OK, Места работы: OK, Обшая оценка извлечения: 95

--- Файл 3/10: 2.docx ---
📄 Парсим файл...
✅ Извлечено 4359 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: OK, Места работы: OK, Обшая оценка извлечения: 95

--- Файл 4/10: 3.docx ---
📄 Парсим файл...
✅ Извлечено 5566 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: FAIL, Места работы: FAIL, Обшая оценка извлечения: 85

--- Файл 5/10: 4.docx ---
📄 Парсим файл...
✅ Извлечено 1170 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: FAIL, Места работы: OK, Обшая оценка извлечения: 90

--- Файл 6/10: 5.docx ---
📄 Парсим файл...
✅ Извлечено 2835 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: OK, Места работы: FAIL, Обшая оценка извлечения: 90

--- Файл 7/10: 6.docx ---
📄 Парсим файл...
✅ Извлечено 2468 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: FAIL, Места работы: FAIL, Обшая оценка извлечения: 85

--- Файл 8/10: 7.docx ---
📄 Парсим файл...
✅ Извлечено 5857 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: OK, Места работы: OK, Обшая оценка извлечения: 95

--- Файл 9/10: 8.docx ---
📄 Парсим файл...
✅ Извлечено 3938 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: FAIL, Места работы: OK, Обшая оценка извлечения: 90

--- Файл 10/10: 9.docx ---
📄 Парсим файл...
✅ Извлечено 4719 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
  Все поля: OK, Все навыки: FAIL, Места работы: FAIL, Обшая оценка извлечения: 85

📊 Итог по 10 тестам:
✅ Все ли поля на месте: 10/10
✅ Все ли навыки извлечены: 4/10
✅ Весь ли опыт извлечен: FAIL/10
✅ Оценки извлечения: 10/10
✅ Средняя оценка извлечения: 89.5
✅ Все условия: 3/10

In [ ]:
PS C:\Users\67181\OneDrive\Dokumenty\bachelor-2025-team-team> & C:/Users/67181/AppData/Local/Programs/Python/Python312/python.exe c:/Users/67181/OneDrive/Dokumenty/bachelor-2025-team-team/ml/evaluation/test_with_llm.py
📄 Найден файл: resume_analysis_1.json
📄 Найден файл: resume_analysis_10.json
📄 Найден файл: resume_analysis_2.json
📄 Найден файл: resume_analysis_3.json
📄 Найден файл: resume_analysis_4.json
📄 Найден файл: resume_analysis_5.json
📄 Найден файл: resume_analysis_6.json
📄 Найден файл: resume_analysis_7.json
📄 Найден файл: resume_analysis_8.json
📄 Найден файл: resume_analysis_9.json
✅ Загружены данные вакансии: Не указана

--- Файл 1/10: resume_analysis_1.json ---
✅ Загружен анализ резюме (4.4 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 86.5/100
Соответствие должности: 70
Вес криетрия должности:0.15
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 90
Вес криетрия опыта: 0.2
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 85
Вес криетрия дополнительных критерией: 0.45
Общая оценка: 85
Соотвествие рекомедаций: 90


--- Файл 2/10: resume_analysis_10.json ---
✅ Загружен анализ резюме (4.0 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 87.5/100
Соответствие должности: 90
Вес криетрия должности:0.15
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 100
Вес криетрия опыта: 0.2
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 85
Вес криетрия дополнительных критерией: 0.45
Общая оценка: 90
Соотвествие рекомедаций: 95


--- Файл 3/10: resume_analysis_2.json ---
✅ Загружен анализ резюме (3.4 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 87.5/100
Соответствие должности: 90
Вес криетрия должности:0.15
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 95
Вес криетрия опыта: 0.25
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 80
Вес криетрия дополнительных критерией: 0.4
Общая оценка: 85
Соотвествие рекомедаций: 90


--- Файл 4/10: resume_analysis_3.json ---
✅ Загружен анализ резюме (8.9 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 87.5/100
Соответствие должности: 90
Вес криетрия должности:0.15
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 100
Вес криетрия опыта: 0.2
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 85
Вес криетрия дополнительных критерией: 0.45
Общая оценка: 90
Соотвествие рекомедаций: 95


--- Файл 5/10: resume_analysis_4.json ---
✅ Загружен анализ резюме (1.8 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 85.5/100
Соответствие должности: 90
Вес криетрия должности:0.15
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 95
Вес криетрия опыта: 0.25
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 80
Вес криетрия дополнительных критерией: 0.4
Общая оценка: 85
Соотвествие рекомедаций: 90


--- Файл 6/10: resume_analysis_5.json ---
✅ Загружен анализ резюме (4.4 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 87.5/100
Соответствие должности: 85
Вес криетрия должности:0.15
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 95
Вес криетрия опыта: 0.25
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 80
Вес криетрия дополнительных критерией: 0.4
Общая оценка: 90
Соотвествие рекомедаций: 95


--- Файл 7/10: resume_analysis_6.json ---
✅ Загружен анализ резюме (3.2 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 87.3/100
Соответствие должности: 90
Вес криетрия должности:0.2
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 95
Вес криетрия опыта: 0.25
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 85
Вес криетрия дополнительных критерией: 0.35
Общая оценка: 90
Соотвествие рекомедаций: 95


--- Файл 8/10: resume_analysis_7.json ---
✅ Загружен анализ резюме (8.1 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 87.5/100
Соответствие должности: 90
Вес криетрия должности:0.2
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 95
Вес криетрия опыта: 0.25
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 80
Вес криетрия дополнительных критерией: 0.35
Общая оценка: 90
Соотвествие рекомедаций: 95


--- Файл 9/10: resume_analysis_8.json ---
✅ Загружен анализ резюме (7.2 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 97/100
Соответствие должности: 90
Вес криетрия должности:0.2
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 100
Вес криетрия опыта: 0.25
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 95
Вес криетрия дополнительных критерией: 0.35
Общая оценка: 98
Соотвествие рекомедаций: 100


--- Файл 10/10: resume_analysis_9.json ---
✅ Загружен анализ резюме (6.4 KB)
🎯 Оцениваем соответствие кандидата вакансии...
✅ Оценка соответствия: 87.5/100
Соответствие должности: 90
Вес криетрия должности:0.15
Соответствие уровня образования: 100
Вес криетрия уровня образования: 0.1
Соответствие опыту: 95
Вес криетрия опыта: 0.25
Соответствие графику работы: 100
Вес криетрия графика работы: 0.05
Соответствие формату работы: 100
Вес криетрия формата работы: 0.05
Соответствие дополнительным критериям: 80
Вес криетрия дополнительных критерией: 0.4
Общая оценка: 85
Соотвествие рекомедаций: 90

PS C:\Users\67181\OneDrive\Dokumenty\bachelor-2025-team-team> 

In [ ]:
PS C:\Users\67181\OneDrive\Dokumenty\bachelor-2025-team-team> & C:/Users/67181/AppData/Local/Programs/Python/Python312/python.exe c:/Users/67181/OneDrive/Dokumenty/bachelor-2025-team-team/ml/evaluation/test_with_llm.py
✅ Загружены данные вакансии: Не указана

--- Файл 1/10: 1.docx ---
📄 Парсим файл...
✅ Извлечено 3702 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 95/100 (OK)
  • Релевантность вакансии: 95/100 (OK)
  • Конкретность вопросов: 90/100 (OK)
  • Разнообразие вопросов: 85/100 (OK)
  • Практическая полезность: 92/100 (OK)
  • Повторений: 1
  • Время проверки: 38.703 сек

🏆 Лучшие вопросы:
  1. Как вы организуете управление состоянием в приложениях на React? Какие инструменты и подходы вы используете?
  2. Расскажите о случае, когда вам пришлось проводить Code Review. Как вы это делали и какие результаты получили?
  3. Как вы интегрируете бэкенд API в свои фронтенд-приложения?

⚠️ Вопросы для улучшения:
  1. Почему вы решили стать фронтенд-разработчиком и что вас мотивирует в этой профессии?
  2. Опишите ситуацию, когда вам пришлось работать в команде над сложным проектом. Какие были ваши задачи и как вы с ними справились?        
  3. Какие у вас ожидания от работы в нашей компании и как вы видите свое развитие здесь?

--- Файл 2/10: 10.docx ---
📄 Парсим файл...
✅ Извлечено 3740 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 95/100 (OK)
  • Релевантность вакансии: 90/100 (OK)
  • Конкретность вопросов: 85/100 (OK)
  • Разнообразие вопросов: 90/100 (OK)
  • Практическая полезность: 92/100 (OK)
  • Повторений: 1
  • Время проверки: 25.826 сек

🏆 Лучшие вопросы:
  1. Расскажите о случае, когда вам пришлось внедрять микросервис в существующее приложение. С какими трудностями вы столкнулись и как их преодолели?
  2. Какие инструменты и методы вы используете для тестирования ваших React-приложений?
  3. Как вы работаете с GraphQL и gRPC в ваших проектах?

⚠️ Вопросы для улучшения:
  1. Как вы предпочитаете организовывать свою работу в команде?
  2. Почему вы решили сменить работу и что вас привлекает в нашей компании?
  3. Какие у вас карьерные цели на ближайшие 5 лет?

--- Файл 3/10: 2.docx ---
📄 Парсим файл...
✅ Извлечено 4359 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 90/100 (OK)
  • Релевантность вакансии: 95/100 (OK)
  • Конкретность вопросов: 85/100 (OK)
  • Разнообразие вопросов: 85/100 (OK)
  • Практическая полезность: 90/100 (OK)
  • Повторений: 1
  • Время проверки: 36.807 сек

🏆 Лучшие вопросы:
  1. Расскажите, как вы использовали Redux Toolkit и Redux Saga в своих проектах. В чем разница между ними?
  2. Как вы тестируете свои приложения? Какие инструменты и подходы используете?
  3. Расскажите о самом сложном техническом вызове, с которым вы столкнулись, и как вы его преодолели.

⚠️ Вопросы для улучшения:
  1. Как вы организуете структуру проекта на React? Какие подходы и инструменты используете?
  2. Как вы обеспечиваете безопасность в своих приложениях? Какие методы и инструменты используете?
  3. Какие ценности для вас важны в компании? Как вы их соблюдаете?

--- Файл 4/10: 3.docx ---
📄 Парсим файл...
✅ Извлечено 5566 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 95/100 (OK)
  • Релевантность вакансии: 90/100 (OK)
  • Конкретность вопросов: 90/100 (OK)
  • Разнообразие вопросов: 85/100 (OK)
  • Практическая полезность: 92/100 (OK)
  • Повторений: 1
  • Время проверки: 69.585 сек

🏆 Лучшие вопросы:
  1. Расскажите, как вы внедряли методологию FSD в проекте TSP. С какими трудностями столкнулись и как их преодолели?
  2. Как вы реализовали кастомный движок диаграмм на базе SVG? Какие технологии и подходы использовали?
  3. Расскажите о случае, когда вам пришлось принимать самостоятельное решение, которое повлияло на проект. Как вы поступили и каков был результат?

⚠️ Вопросы для улучшения:
  1. Какие инструменты сборки и тестирования вы используете в своих проектах?
  2. Какие у вас ожидания относительно заработной платы и условий работы?
  3. Как вы оцениваете свой уровень владения английским языком? Достаточно ли его для чтения технической документации?

--- Файл 5/10: 4.docx ---
📄 Парсим файл...
✅ Извлечено 1170 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 95/100 (OK)
  • Релевантность вакансии: 85/100 (OK)
  • Конкретность вопросов: 90/100 (OK)
  • Разнообразие вопросов: 85/100 (OK)
  • Практическая полезность: 90/100 (OK)
  • Повторений: 1
  • Время проверки: 23.654 сек

🏆 Лучшие вопросы:
  1. Как вы осуществляли миграцию с Vue 2 на Vue 3? С какими трудностями столкнулись и как их преодолели?
  2. Расскажите о случае, когда вам пришлось работать в команде над сложным проектом. Какую роль вы выполняли и как справлялись с трудностями?
  3. Объясните, пожалуйста, принципы работы с TypeScript. Какие типы данных и интерфейсы вы используете чаще всего?

⚠️ Вопросы для улучшения:
  1. У вас нет опыта работы с React. Как вы планируете освоить эту технологию?
  2. Какие у вас ожидания относительно заработной платы и условий работы?
  3. Как вы относитесь к гибридному формату работы? Какие преимущества и недостатки видите в таком формате?

--- Файл 6/10: 5.docx ---
📄 Парсим файл...
✅ Извлечено 2835 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 90/100 (OK)
  • Релевантность вакансии: 85/100 (OK)
  • Конкретность вопросов: 85/100 (OK)
  • Разнообразие вопросов: 80/100 (OK)
  • Практическая полезность: 88/100 (OK)
  • Повторений: 1
  • Время проверки: 22.102 сек

🏆 Лучшие вопросы:
  1. Расскажите, как вы решали задачу по оптимизации скорости работы сайта. Какие инструменты и методы вы использовали?
  2. Какие у вас есть опыт работы с React и TypeScript? Если опыта нет, как вы планируете освоить эти технологии?
  3. Как вы работаете с чужым кодом? Приведите пример, когда вам приходилось разбираться в чужом коде и вносить в него изменения.

⚠️ Вопросы для улучшения:
  1. Почему вы решили сменить сферу деятельности с юридической на IT?
  2. Какие методы и инструменты вы используете для тестирования сайтов?
  3. Какие у вас ожидания от новой работы и компании?

--- Файл 7/10: 6.docx ---
📄 Парсим файл...
✅ Извлечено 2468 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 95/100 (OK)
  • Релевантность вакансии: 95/100 (OK)
  • Конкретность вопросов: 90/100 (OK)
  • Разнообразие вопросов: 90/100 (OK)
  • Практическая полезность: 92/100 (OK)
  • Повторений: 0
  • Время проверки: 48.969 сек

🏆 Лучшие вопросы:
  1. Расскажите, как вы оптимизировали приложение с помощью memo, useMemo, useCallback, lazy-loading и debounce. Приведите примеры из вашего опыта.
  2. Объясните, как вы реализовали перехватчик запросов и работу с CSRF-токенами и cookie.
  3. Как вы тестируете свои приложения? Какие инструменты и подходы используете?

⚠️ Вопросы для улучшения:
  1. Почему вы решили стать фронтенд-разработчиком и что вас мотивирует в этой профессии?
  2. Какие у вас карьерные цели на ближайшие 5 лет?
  3. Как вы относитесь к гибридному формату работы? Какие плюсы и минусы вы видите в таком формате?

--- Файл 8/10: 7.docx ---
📄 Парсим файл...
✅ Извлечено 5857 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 95/100 (OK)
  • Релевантность вакансии: 95/100 (OK)
  • Конкретность вопросов: 90/100 (OK)
  • Разнообразие вопросов: 90/100 (OK)
  • Практическая полезность: 92/100 (OK)
  • Повторений: 1
  • Время проверки: 36.53 сек

🏆 Лучшие вопросы:
  1. Расскажите, как вы использовали Redux в своих проектах. Какие middleware вы применяли и для чего?
  2. Представьте, что вам нужно оптимизировать производительность React-приложения. Какие шаги вы предпримете?
  3. Расскажите о случае, когда вам пришлось решать сложную техническую проблему в команде. Как вы подходили к решению?

⚠️ Вопросы для улучшения:
  1. Почему вы решили сменить сферу деятельности с системного администрирования на разработку?
  2. Как вы обеспечиваете качество кода в своих проектах? Какие практики и инструменты вы используете?
  3. Как вы работаете с API в React-приложениях? Какие библиотеки или подходы вы используете?

--- Файл 9/10: 8.docx ---
📄 Парсим файл...
✅ Извлечено 3938 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 90/100 (OK)
  • Релевантность вакансии: 95/100 (OK)
  • Конкретность вопросов: 85/100 (OK)
  • Разнообразие вопросов: 85/100 (OK)
  • Практическая полезность: 90/100 (OK)
  • Повторений: 1
  • Время проверки: 28.157 сек

🏆 Лучшие вопросы:
  1. Как вы реализовывали real-time обновления через WebSocket в своих проектах?
  2. Расскажите о самом сложном проекте, над которым вы работали, и как вы справлялись с трудностями?
  3. Как вы организовывали процесс CI/CD в своих проектах?

⚠️ Вопросы для улучшения:
  1. Какие инструменты сборки и тестирования вы использовали в своих проектах?
  2. Почему вы решили сменить работу и что вас привлекает в нашей компании?
  3. Какие принципы UI/UX вы учитывали при разработке интерфейсов?

--- Файл 10/10: 9.docx ---
📄 Парсим файл...
✅ Извлечено 4719 символов
🔍 Анализируем резюме...
✅ Анализ завершен успешно
📝 Генерируем вопросы для интервью...
📊 Результаты проверки:
  • Релевантность резюме: 88/100 (OK)
  • Релевантность вакансии: 85/100 (OK)
  • Конкретность вопросов: 80/100 (OK)
  • Разнообразие вопросов: 85/100 (OK)
  • Практическая полезность: 87/100 (OK)
  • Повторений: 1
  • Время проверки: 42.611 сек

🏆 Лучшие вопросы:
  1. Как вы работаете с API? Можете привести пример интеграции сторонних сервисов по API в ваших проектах?
  2. Расскажите о случае, когда вам приходилось быстро осваивать новые технологии для реализации проекта. Как вы с этим справились?
  3. Как вы организуете процесс сборки и тестирования фронтенд-проектов? Какие инструменты используете?

⚠️ Вопросы для улучшения:
  1. У вас есть опыт работы с React?
  2. Объясните, пожалуйста, разницу между Angular и Vue.js. С каким из этих фреймворков вы предпочитаете работать и почему?
  3. Как вы обеспечиваете адаптивность верстки? Какие инструменты и подходы используете?

============================================================
📊 ИТОГОВАЯ СТАТИСТИКА ПО ГЕНЕРАЦИИ ВОПРОСОВ
============================================================
Всего протестировано: 10 резюме

✅ Релевантность резюме: 10/10
   Средняя оценка: 92.8/100

✅ Релевантность вакансии: 10/10
   Средняя оценка: 91.0/100

✅ Конкретность вопросов: 10/10
   Средняя оценка: 87.0/100

✅ Разнообразие вопросов: 10/10
   Средняя оценка: 86.0/100

✅ Практическая полезность: 10/10
   Средняя оценка: 90.5/100

🎯 Все критерии выполнены: 10/10

📈 Общие средние оценки:
  • Релевантность резюме: 92.8/100
  • Релевантность вакансии: 91.0/100
  • Конкретность: 87.0/100
  • Разнообразие: 86.0/100
  • Полезность: 90.5/100